# WISDM Fenstergrößen-Experiment

**Fragestellung:** Ab welcher Reaktionszeit wird die Aktivitätserkennung unzuverlässig?

Wir untersuchen, wie sich verschiedene Klassifikatoren (k-NN, SVM, Random Forest, MLP) verhalten, wenn das Smartphone nur wenig Zeit hat, eine Aktivität zu erkennen.

## 1. Setup

In [ ]:
from pathlib import PurePosixPath
from timeit import default_repeat

# Dependencies (in Colab bereits vorinstalliert)
import numpy as np
import pandas as pd
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import urllib.request
import tarfile
warnings.filterwarnings('ignore')

print("Setup abgeschlossen!")

## 2. Konfiguration

In [ ]:
# =============================================================================
# KONFIGURATION - HIER ANPASSEN
# =============================================================================

# Output-Verzeichnis für Plots und Ergebnisse
OUTPUT_DIR = "output"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fenstergrößen zum Testen (bei ~20 Hz Samplerate)
WINDOW_SIZES = [50, 75, 100, 150, 200, 300, 400]
# Entspricht ca.: 2.5s, 3.75s, 5s, 7.5s, 10s, 15s, 20s

STEP_RATIO = 0.5  # 50% Overlap

# Datensatz-Pfade
TRAIN_DATA_PATH = "data/WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt"  # AR für Training
TEST_DATA_PATH = "data/WISDM_at_v2.0/WISDM_at_v2.0_raw.txt"  # AT für Test (optional)

# Auf AT-Datensatz testen? (Falls False, wird Train/Test-Split auf AR gemacht)
USE_AT_FOR_TESTING = True  # Auf True setzen wenn AT-Datensatz vorhanden

SAMPLE_RATE_HZ = 20  # Ungefähre Samplerate des Datensatzes

# Reaktionszeiten anzeigen
print("Fenstergrößen und entsprechende Reaktionszeiten:")
for w in WINDOW_SIZES:
    print(f"  {w:3d} Samples = {w/SAMPLE_RATE_HZ:5.2f} Sekunden")

## 3. Datensatz laden

Prüft automatisch ob die Daten lokal vorhanden sind. Falls nicht, werden sie heruntergeladen.

In [ ]:


# =============================================================================
# PFADE KONFIGURATION
# =============================================================================

# Basis-Verzeichnis für Datensätze (anpassen falls nötig)
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

# Datensatz-Pfade
TRAIN_DATA_PATH = f"{DATA_DIR}/WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt"
TEST_DATA_PATH = f"{DATA_DIR}/WISDM_at_v2.0/WISDM_at_v2.0_raw.txt"

# Download-URLs
WISDM_AR_URL = "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz"
WISDM_AT_URL = "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_at_latest.tar.gz"


# Sets sind unterschiedlich gepackt, directory traversal nutzen
def deepest_dir(members):
    dirs = set()
    for m in members:
        p = PurePosixPath(m.name)
        if m.isdir():
            dirs.add(p)
        elif p.parent != PurePosixPath("."):
            dirs.add(p.parent)
        return max(dirs, key =lambda p: len(p.parts))


def download_and_extract(url, target_dir, name):
    """Lädt Datensatz herunter und entpackt ihn."""
    archive_path = os.path.join(target_dir, f"{name}.tar.gz")
    
    print(f"Lade {name} herunter...")
    urllib.request.urlretrieve(url, archive_path)
    
    print(f"Entpacke {name}...")
    with tarfile.open(archive_path, 'r:gz') as tar:
        members = tar.getmembers()
        deepest = deepest_dir(members)
        selected=[]
        parent = deepest.parent
        for m in members:
            p = PurePosixPath(m.name)
            if p == deepest or deepest in p.parents:
                m.name = str(p.relative_to(parent))
                selected.append(m)
        tar.extractall(path=target_dir, members= selected)
    
    # Archiv löschen
    os.remove(archive_path)
    !ls
    print(f"{name} erfolgreich heruntergeladen!")

# WISDM AR (Trainingsdaten)
if os.path.exists(TRAIN_DATA_PATH):
    print(f"Trainingsdaten gefunden: {TRAIN_DATA_PATH}")
else:
    print(f"Trainingsdaten nicht gefunden")
    download_and_extract(WISDM_AR_URL, DATA_DIR, "WISDM_ar")

# WISDM AT (Testdaten)
if USE_AT_FOR_TESTING:
    if os.path.exists(TEST_DATA_PATH):
        print(f"Testdaten gefunden: {TEST_DATA_PATH}")
    else:
        print(f"Testdaten nicht gefunden")
        download_and_extract(WISDM_AT_URL, DATA_DIR, "WISDM_at")

## 4. Hilfsfunktionen

In [ ]:
def load_wisdm_data(filepath):
    """Lädt WISDM Rohdaten aus Textdatei"""
    data = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip().rstrip(';')
            if not line:
                continue
            parts = line.split(',')
            if len(parts) == 6:
                try:
                    user = int(parts[0])
                    activity = parts[1].strip()
                    timestamp = int(parts[2])
                    x = float(parts[3])
                    y = float(parts[4])
                    z = float(parts[5].rstrip(';'))
                    data.append([user, activity, timestamp, x, y, z])
                except (ValueError, IndexError):
                    continue
    return pd.DataFrame(data, columns=['user', 'activity', 'timestamp', 'x', 'y', 'z'])


def extract_features_vectorized(windows):
    """
    Vektorisierte Feature-Extraktion für alle Fenster gleichzeitig.
    Deutlich schneller als die Einzelfenster-Variante.
    
    Args:
        windows: NumPy Array mit Shape (n_windows, window_size, 3) für x, y, z
    
    Returns:
        features: NumPy Array mit Shape (n_windows, 33)
    """
    feature_list = []
    
    # Features pro Achse (x=0, y=1, z=2)
    for axis in range(3):
        data = windows[:, :, axis]
        
        feature_list.append(np.mean(data, axis=1))
        feature_list.append(np.std(data, axis=1))
        feature_list.append(np.min(data, axis=1))
        feature_list.append(np.max(data, axis=1))
        feature_list.append(np.max(data, axis=1) - np.min(data, axis=1))  # Range
        feature_list.append(np.median(data, axis=1))
        feature_list.append(stats.skew(data, axis=1))
        feature_list.append(stats.kurtosis(data, axis=1))
        feature_list.append(np.sqrt(np.mean(data**2, axis=1)))  # RMS
        feature_list.append(np.sum(np.abs(np.diff(data, axis=1)), axis=1))  # Abs Diff Sum
    
    # Magnitude Features
    magnitude = np.sqrt(windows[:,:,0]**2 + windows[:,:,1]**2 + windows[:,:,2]**2)
    feature_list.append(np.mean(magnitude, axis=1))
    feature_list.append(np.std(magnitude, axis=1))
    feature_list.append(np.max(magnitude, axis=1) - np.min(magnitude, axis=1))
    
    return np.array(feature_list).T


def create_dataset(df, window_size, step_size):
    """
    Erstellt Feature-Datensatz aus Rohdaten mit gegebener Fenstergröße.
    Verwendet vektorisierte Feature-Extraktion für bessere Performance.
    """
    X_list = []
    y_list = []
    
    for (user, activity), group in df.groupby(['user', 'activity']):
        group = group.sort_values('timestamp').reset_index(drop=True)
        
        # Daten als NumPy Array
        data = group[['x', 'y', 'z']].values
        n_samples = len(data)
        
        # Anzahl möglicher Fenster
        n_windows = (n_samples - window_size) // step_size + 1
        
        if n_windows <= 0:
            continue
        
        # Alle Fenster auf einmal erstellen (vektorisiert)
        indices = np.arange(n_windows)[:, None] * step_size + np.arange(window_size)
        windows = data[indices]  # Shape: (n_windows, window_size, 3)
        
        # Features vektorisiert extrahieren
        features = extract_features_vectorized(windows)
        
        X_list.append(features)
        y_list.extend([activity] * n_windows)
    
    X = np.vstack(X_list)
    y = np.array(y_list)
    
    # NaN/Inf bereinigen
    mask = ~(np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1))
    X = X[mask]
    y = y[mask]
    
    return X, y


def get_classifiers():
    """Gibt Dictionary mit allen Klassifikatoren zurück"""
    return {
        "k-NN": KNeighborsClassifier(n_neighbors=5),
        "SVM": SVC(kernel='rbf', C=1.0, gamma='scale'),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "MLP": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, early_stopping=True)
    }

print("Hilfsfunktionen geladen!")

## 5. Daten laden und erkunden

In [ ]:
# Trainingsdaten laden
print("Lade Trainingsdaten (WISDM AR)...")
df_train = load_wisdm_data(TRAIN_DATA_PATH)
print(f"Geladene Datenpunkte: {len(df_train):,}")

# Optional: AT-Datensatz laden
if USE_AT_FOR_TESTING:
    print("\nLade Testdaten (WISDM AT)...")
    df_test = load_wisdm_data(TEST_DATA_PATH)
    print(f"Geladene Datenpunkte: {len(df_test):,}")

In [ ]:
# Datensatz erkunden
print("Aktivitäten im Datensatz:")
print(df_train['activity'].value_counts())

print(f"\nAnzahl Probanden: {df_train['user'].nunique()}")

In [ ]:
# Visualisierung: Beispiel-Rohdaten
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

activities = df_train['activity'].unique()
for idx, activity in enumerate(activities[:6]):
    ax = axes[idx // 3, idx % 3]
    sample = df_train[df_train['activity'] == activity].head(200)
    ax.plot(sample['x'].values, label='x', alpha=0.8)
    ax.plot(sample['y'].values, label='y', alpha=0.8)
    ax.plot(sample['z'].values, label='z', alpha=0.8)
    ax.set_title(activity)
    ax.legend(loc='upper right')
    ax.set_xlabel('Sample')
    ax.set_ylabel('Beschleunigung')

plt.tight_layout()
plt.suptitle('Accelerometer-Rohdaten pro Aktivität', y=1.02, fontsize=14)
plt.show()

## 6. Experiment: Fenstergrößen-Vergleich

In [ ]:
print("=" * 70)
print("EXPERIMENT: Fenstergrößen-Vergleich")
print("=" * 70)
print(f"\nFenstergrößen: {WINDOW_SIZES}")
print(f"Reaktionszeiten: {[f'{w/SAMPLE_RATE_HZ:.1f}s' for w in WINDOW_SIZES]}")

results = []

for window_size in WINDOW_SIZES:
    step_size = int(window_size * STEP_RATIO)
    reaction_time = window_size / SAMPLE_RATE_HZ
    
    print(f"\n{'='*50}")
    print(f"Fenstergröße: {window_size} ({reaction_time:.1f}s Reaktionszeit)")
    print(f"{'='*50}")
    
    # Features extrahieren
    print("Feature-Extraktion...")
    X_train_full, y_train_full = create_dataset(df_train, window_size, step_size)
    
    if USE_AT_FOR_TESTING:
        X_test, y_test = create_dataset(df_test, window_size, step_size)
        X_train, y_train = X_train_full, y_train_full
        # Label Encoding
        le = LabelEncoder()
        le.fit(np.concatenate([y_train, y_test]))
        y_train = le.transform(y_train)
        y_test = le.transform(y_test)
    else:
        # Train/Test Split auf AR-Datensatz
        le = LabelEncoder()
        y_encoded = le.fit_transform(y_train_full)
        X_train, X_test, y_train, y_test = train_test_split(
            X_train_full, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
        )
    
    print(f"Samples - Train: {len(X_train):,}, Test: {len(X_test):,}")
    
    # Skalierung
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Klassifikatoren trainieren und evaluieren
    classifiers = get_classifiers()
    
    print("\nErgebnisse:")
    for name, clf in classifiers.items():
        start = time.time()
        
        if name == "Random Forest":
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
        else:
            clf.fit(X_train_scaled, y_train)
            y_pred = clf.predict(X_test_scaled)
        
        elapsed = time.time() - start
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results.append({
            'window_size': window_size,
            'reaction_time_s': reaction_time,
            'classifier': name,
            'accuracy': acc,
            'f1_score': f1,
            'train_time': elapsed,
            'n_train': len(X_train),
            'n_test': len(X_test)
        })
        
        print(f"  {name:15s}: {100*acc:5.1f}% (F1: {f1:.3f})")

print("\n" + "=" * 70)
print("Experiment abgeschlossen!")
print("=" * 70)

  k-NN           :  59.2% (F1: 0.599)


## 7. Experiment B: Konstante Sample-Anzahl

Im vorherigen Experiment hatten kleinere Fenster automatisch mehr Trainingssamples. Jetzt kontrollieren wir dafür: **Alle Fenstergrößen bekommen die gleiche Anzahl Samples.**

So sehen wir isoliert, wie sich die Fenstergröße (= Informationsgehalt pro Sample) auswirkt – unabhängig von der Trainingssetgröße.

In [ ]:
print("=" * 70)
print("EXPERIMENT B: Konstante Sample-Anzahl")
print("=" * 70)

# Zuerst herausfinden, wie viele Samples die größte Fenstergröße erzeugt
max_window = max(WINDOW_SIZES)
step_size_max = int(max_window * STEP_RATIO)
X_max, y_max = create_dataset(df_train, max_window, step_size_max)

# Das ist unsere Referenz-Samplegröße
FIXED_SAMPLE_COUNT = len(X_max)
print(f"\nReferenz: Fenstergröße {max_window} erzeugt {FIXED_SAMPLE_COUNT:,} Samples")
print(f"Alle anderen Fenstergrößen werden auf diese Anzahl reduziert.")
print(f"\nFenstergrößen: {WINDOW_SIZES}")

results_fixed = []

for window_size in WINDOW_SIZES:
    step_size = int(window_size * STEP_RATIO)
    reaction_time = window_size / SAMPLE_RATE_HZ
    
    print(f"\n{'='*50}")
    print(f"Fenstergröße: {window_size} ({reaction_time:.1f}s Reaktionszeit)")
    print(f"{'='*50}")
    
    # Features extrahieren
    X_full, y_full = create_dataset(df_train, window_size, step_size)
    print(f"Ursprüngliche Samples: {len(X_full):,}")
    
    # Auf FIXED_SAMPLE_COUNT reduzieren (stratified sampling)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_full)
    
    if len(X_full) > FIXED_SAMPLE_COUNT:
        # Reduzieren durch stratified sampling
        keep_ratio = FIXED_SAMPLE_COUNT / len(X_full)
        X_reduced, _, y_reduced, _ = train_test_split(
            X_full, y_encoded, train_size=keep_ratio, random_state=42, stratify=y_encoded
        )
    else:
        X_reduced, y_reduced = X_full, y_encoded
    
    print(f"Reduziert auf: {len(X_reduced):,} Samples")
    
    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_reduced, y_reduced, test_size=0.3, random_state=42, stratify=y_reduced
    )
    
    print(f"Train: {len(X_train):,}, Test: {len(X_test):,}")
    
    # Skalierung
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Klassifikatoren
    classifiers = get_classifiers()
    
    print("\nErgebnisse:")
    for name, clf in classifiers.items():
        start = time.time()
        
        if name == "Random Forest":
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
        else:
            clf.fit(X_train_scaled, y_train)
            y_pred = clf.predict(X_test_scaled)
        
        elapsed = time.time() - start
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        results_fixed.append({
            'window_size': window_size,
            'reaction_time_s': reaction_time,
            'classifier': name,
            'accuracy': acc,
            'f1_score': f1,
            'train_time': elapsed,
            'n_train': len(X_train),
            'n_test': len(X_test)
        })
        
        print(f"  {name:15s}: {100*acc:5.1f}% (F1: {f1:.3f})")

results_fixed_df = pd.DataFrame(results_fixed)
print("\n" + "=" * 70)
print("Experiment B abgeschlossen!")
print("=" * 70)

In [ ]:
# Zusammenfassung Experiment B
print("ZUSAMMENFASSUNG EXPERIMENT B: Konstante Sample-Anzahl\n")
pivot_fixed = results_fixed_df.pivot(index='window_size', columns='classifier', values='accuracy')
pivot_fixed['Reaktionszeit (s)'] = pivot_fixed.index / SAMPLE_RATE_HZ
pivot_fixed = pivot_fixed[['Reaktionszeit (s)', 'k-NN', 'SVM', 'Random Forest', 'MLP']]
print(pivot_fixed.round(4).to_string())

In [ ]:
# Vergleichsplot: Experiment A vs B
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'k-NN': '#1f77b4', 'SVM': '#ff7f0e', 'Random Forest': '#2ca02c', 'MLP': '#d62728'}

# Experiment A (variable Sample-Anzahl)
ax1 = axes[0]
for classifier in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    data = results_df[results_df['classifier'] == classifier]
    ax1.plot(data['reaction_time_s'], data['accuracy'], 
             marker='o', label=classifier, linewidth=2, markersize=8, color=colors[classifier])

ax1.set_xlabel('Reaktionszeit (Sekunden)', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Experiment A: Variable Sample-Anzahl', fontsize=13)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.5, 1.0)

# Experiment B (konstante Sample-Anzahl)
ax2 = axes[1]
for classifier in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    data = results_fixed_df[results_fixed_df['classifier'] == classifier]
    ax2.plot(data['reaction_time_s'], data['accuracy'], 
             marker='o', label=classifier, linewidth=2, markersize=8, color=colors[classifier])

ax2.set_xlabel('Reaktionszeit (Sekunden)', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title(f'Experiment B: Konstante Sample-Anzahl ({FIXED_SAMPLE_COUNT:,})', fontsize=13)
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/experiment_comparison.png', dpi=150)
plt.show()

print(f"Plot gespeichert: {OUTPUT_DIR}/experiment_comparison.png")

In [ ]:
# Direkter Vergleich: Wie viel macht die zusätzliche Datenmenge aus?
print("VERGLEICH: Einfluss der Trainingsset-Größe\n")
print("Differenz (Experiment A - Experiment B) bei kürzester Reaktionszeit:")
print("(Positiv = mehr Daten helfen, Negativ = mehr Daten schaden)\n")

min_window = min(WINDOW_SIZES)

for clf in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    acc_a = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == min_window)]['accuracy'].values[0]
    acc_b = results_fixed_df[(results_fixed_df['classifier'] == clf) & (results_fixed_df['window_size'] == min_window)]['accuracy'].values[0]
    diff = acc_a - acc_b
    
    n_train_a = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == min_window)]['n_train'].values[0]
    n_train_b = results_fixed_df[(results_fixed_df['classifier'] == clf) & (results_fixed_df['window_size'] == min_window)]['n_train'].values[0]
    
    print(f"  {clf:15s}: {100*diff:+.2f} Prozentpunkte")
    print(f"                   (A: {n_train_a:,} Samples → {100*acc_a:.1f}% | B: {n_train_b:,} Samples → {100*acc_b:.1f}%)")

## 8. Ergebnisse Experiment A

In [ ]:
# Ergebnisse als DataFrame
results_df = pd.DataFrame(results)

# Pivot-Tabelle
print("ZUSAMMENFASSUNG: Accuracy nach Fenstergröße\n")
pivot = results_df.pivot(index='window_size', columns='classifier', values='accuracy')
pivot['Reaktionszeit (s)'] = pivot.index / SAMPLE_RATE_HZ
pivot = pivot[['Reaktionszeit (s)', 'k-NN', 'SVM', 'Random Forest', 'MLP']]
print(pivot.round(4).to_string())

In [ ]:
# Hauptplot: Accuracy vs. Reaktionszeit
plt.figure(figsize=(10, 6))

colors = {'k-NN': '#1f77b4', 'SVM': '#ff7f0e', 'Random Forest': '#2ca02c', 'MLP': '#d62728'}

for classifier in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    data = results_df[results_df['classifier'] == classifier]
    plt.plot(data['reaction_time_s'], data['accuracy'], 
             marker='o', label=classifier, linewidth=2, markersize=8, color=colors[classifier])

plt.xlabel('Reaktionszeit (Sekunden)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Klassifikationsgenauigkeit vs. Reaktionszeit', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/window_size_comparison.png', dpi=150)
plt.show()

print(f"Plot gespeichert: {OUTPUT_DIR}/window_size_comparison.png")

In [ ]:
# Heatmap der Ergebnisse
plt.figure(figsize=(10, 6))

pivot_heatmap = results_df.pivot(index='classifier', columns='reaction_time_s', values='accuracy')
sns.heatmap(pivot_heatmap, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.6, vmax=1.0)

plt.xlabel('Reaktionszeit (Sekunden)', fontsize=12)
plt.ylabel('Klassifikator', fontsize=12)
plt.title('Accuracy Heatmap: Klassifikator vs. Reaktionszeit', fontsize=14)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/accuracy_heatmap.png', dpi=150)
plt.show()

## 9. Detailanalyse: Beste und schlechteste Konfiguration

In [ ]:
# Beste Konfiguration
best_idx = results_df['accuracy'].idxmax()
best = results_df.loc[best_idx]
print(f"BESTE KONFIGURATION:")
print(f"  Klassifikator: {best['classifier']}")
print(f"  Fenstergröße: {int(best['window_size'])} ({best['reaction_time_s']:.1f}s)")
print(f"  Accuracy: {100*best['accuracy']:.2f}%")

# Schlechteste Konfiguration
worst_idx = results_df['accuracy'].idxmin()
worst = results_df.loc[worst_idx]
print(f"\nSCHLECHTESTE KONFIGURATION:")
print(f"  Klassifikator: {worst['classifier']}")
print(f"  Fenstergröße: {int(worst['window_size'])} ({worst['reaction_time_s']:.1f}s)")
print(f"  Accuracy: {100*worst['accuracy']:.2f}%")

In [ ]:
# Accuracy-Verlust pro Klassifikator
print("ACCURACY-VERLUST (längste vs. kürzeste Reaktionszeit):\n")

min_window = min(WINDOW_SIZES)
max_window = max(WINDOW_SIZES)

for clf in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    acc_min = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == min_window)]['accuracy'].values[0]
    acc_max = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == max_window)]['accuracy'].values[0]
    diff = acc_max - acc_min
    print(f"  {clf:15s}: {100*acc_min:.1f}% → {100*acc_max:.1f}% ({100*diff:+.1f} Prozentpunkte)")

## 10. Confusion Matrix für ausgewählte Konfiguration

In [ ]:
# Confusion Matrix für beste Klassifikator bei kürzester Reaktionszeit
# (zeigt welche Aktivitäten bei wenig Daten verwechselt werden)

SELECTED_WINDOW = min(WINDOW_SIZES)  # Kürzeste Reaktionszeit
SELECTED_CLASSIFIER = "MLP"  # Hier anpassen

print(f"Confusion Matrix für {SELECTED_CLASSIFIER} bei Fenstergröße {SELECTED_WINDOW}")
print(f"(Reaktionszeit: {SELECTED_WINDOW/SAMPLE_RATE_HZ:.1f}s)\n")

# Daten vorbereiten
step_size = int(SELECTED_WINDOW * STEP_RATIO)
X_full, y_full = create_dataset(df_train, SELECTED_WINDOW, step_size)
le = LabelEncoder()
y_encoded = le.fit_transform(y_full)
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Klassifikator trainieren
clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, early_stopping=True)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

# Confusion Matrix plotten
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Vorhergesagt')
plt.ylabel('Tatsächlich')
plt.title(f'Confusion Matrix: {SELECTED_CLASSIFIER} ({SELECTED_WINDOW/SAMPLE_RATE_HZ:.1f}s Reaktionszeit)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 11. Ergebnisse exportieren

In [ ]:
# CSV speichern
results_df.to_csv(f'{OUTPUT_DIR}/window_size_results_variable.csv', index=False)
results_fixed_df.to_csv(f'{OUTPUT_DIR}/window_size_results_fixed.csv', index=False)
print(f"Ergebnisse gespeichert in {OUTPUT_DIR}/:")
print("  - window_size_results_variable.csv (Experiment A)")
print("  - window_size_results_fixed.csv (Experiment B)")

# Download in Colab
# from google.colab import files
# files.download('window_size_results.csv')
# files.download('window_size_comparison.png')
# files.download('accuracy_heatmap.png')
# files.download('confusion_matrix.png')

## 12. Fazit

In [ ]:
print("=" * 70)
print("FAZIT")
print("=" * 70)

print(f"\nGetestete Reaktionszeiten: {min(WINDOW_SIZES)/SAMPLE_RATE_HZ:.1f}s bis {max(WINDOW_SIZES)/SAMPLE_RATE_HZ:.1f}s")

# Bester Klassifikator pro Fenstergröße
print("\nBester Klassifikator pro Reaktionszeit:")
for ws in WINDOW_SIZES:
    subset = results_df[results_df['window_size'] == ws]
    best = subset.loc[subset['accuracy'].idxmax()]
    print(f"  {ws/SAMPLE_RATE_HZ:5.1f}s: {best['classifier']:15s} ({100*best['accuracy']:.1f}%)")

# Robustester Klassifikator (geringster Accuracy-Verlust)
print("\nRobustheit (Accuracy-Verlust bei kürzester vs. längster Reaktionszeit):")
robustness = []
for clf in ['k-NN', 'SVM', 'Random Forest', 'MLP']:
    acc_min = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == min(WINDOW_SIZES))]['accuracy'].values[0]
    acc_max = results_df[(results_df['classifier'] == clf) & (results_df['window_size'] == max(WINDOW_SIZES))]['accuracy'].values[0]
    diff = acc_max - acc_min
    robustness.append((clf, diff))

robustness.sort(key=lambda x: x[1])
for clf, diff in robustness:
    print(f"  {clf:15s}: {100*diff:+.1f} Prozentpunkte")

print(f"\n→ Robustester Klassifikator: {robustness[0][0]}")